In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

base_url = "https://www.tianqihoubao.com/lishi/dalian/month/{year}{month:02d}.html"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
}

all_data = []

for year in range(2022, 2026): 
    for month in range(1, 13):
        url = base_url.format(year=year, month=month)
        print(f"Catching: {url}")
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.encoding = "utf-8"
            soup = BeautifulSoup(response.text, "html.parser")

            table = soup.find("table", class_="weather-table")
            if table is None:
                print(f"NULL: {url}")
                continue

            rows = table.find("tbody").find_all("tr")
            for row in rows:
                cols = row.find_all("td")
                if len(cols) < 4:
                    continue
                date = cols[0].get_text(strip=True)
                weather = cols[1].get_text(strip=True)
                temp = cols[2].get_text(strip=True)
                wind = cols[3].get_text(strip=True)

                all_data.append({
                    "date": date,
                    "weather": weather,
                    "temp": temp,
                    "wind": wind,
                    "year": year,
                    "month": month
                })
            time.sleep(0.5)
        except Exception as e:
            print(f"Fail: {url}, error: {e}")
            continue

# 保存为CSV文件
df = pd.DataFrame(all_data)
df.to_csv("dalian_weather.csv", index=False, encoding="utf-8")
print("Done. Total rows:", len(df))

Catching: https://www.tianqihoubao.com/lishi/dalian/month/202201.html
Catching: https://www.tianqihoubao.com/lishi/dalian/month/202202.html
Catching: https://www.tianqihoubao.com/lishi/dalian/month/202203.html
Catching: https://www.tianqihoubao.com/lishi/dalian/month/202204.html
Catching: https://www.tianqihoubao.com/lishi/dalian/month/202205.html
Catching: https://www.tianqihoubao.com/lishi/dalian/month/202206.html
Catching: https://www.tianqihoubao.com/lishi/dalian/month/202207.html
Catching: https://www.tianqihoubao.com/lishi/dalian/month/202208.html
Catching: https://www.tianqihoubao.com/lishi/dalian/month/202209.html
Catching: https://www.tianqihoubao.com/lishi/dalian/month/202210.html
Catching: https://www.tianqihoubao.com/lishi/dalian/month/202211.html
Catching: https://www.tianqihoubao.com/lishi/dalian/month/202212.html
Catching: https://www.tianqihoubao.com/lishi/dalian/month/202301.html
Catching: https://www.tianqihoubao.com/lishi/dalian/month/202302.html
Catching: https://ww

In [3]:
dalian_weather = df
dalian_weather[["wind_day", "wind_night"]] = dalian_weather["wind"].str.split('/', expand=True)

dalian_weather["wind_day_level"] = dalian_weather["wind_day"].str.extract(r'(\d+-\d+)')
dalian_weather["wind_night_level"] = dalian_weather["wind_night"].str.extract(r'(\d+-\d+)')
dalian_weather = dalian_weather.dropna(subset=["temp"])
dalian_weather[["temp_high", "temp_low"]] = dalian_weather["temp"].str.extract(r'(-?\d+)℃/(-?\d+)℃')
dalian_weather["temp_low"] = dalian_weather["temp_low"].astype(float)
dalian_weather["temp_high"] = dalian_weather["temp_high"].astype(float)
dalian_weather[["weather_day", "weather_night"]] = dalian_weather["weather"].str.split('/', expand=True)
dalian_weather.drop(columns=["temp", "wind", "wind_day", "wind_night", 'weather'])
dalian_weather.to_csv("cleaned_weather.csv")